# Convert a Jupyter notebook to PDF

This notebook turns another `.ipynb` file into a PDF. It:

- embeds the pictures referenced in markdown cells (`<img src="images/...">` and `![](...)`), so they show up in the PDF
- removes video cells and error tracebacks, which can't be shown in a PDF
- can optionally hide all code, leaving just text, equations, tables, images and outputs
- lets you set the font size of the text and code
- renders the page in a headless Chrome browser, so HTML tables, LaTeX equations and colored text look the same as in Jupyter

**How to use it**

1. Put this notebook **in the same folder** as the notebook you want to convert, so relative image paths like `images/1_a.JPG` can be found.
2. Edit the settings in **Step 2**.
3. Run all cells (**Kernel → Restart & Run All**). The PDF is saved next to your notebook.

## Step 1: Install what's needed (only the first time)

In [1]:
%pip install -q "nbconvert[webpdf]" nbformat pillow
# Downloads a headless Chromium for rendering the PDF (~150 MB, one time only)
!python -m playwright install chromium


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
zsh:1: command not found: python


## Step 2: Settings

In [2]:
INPUT_NOTEBOOK = "Lab 1/Lab1 Microscopy 2026.ipynb"   # notebook to convert (same folder as this one)
OUTPUT_PDF     = None        # None = same name as the input, with .pdf

HIDE_CODE        = True      # True: hide all code cells' input (good for lab reports)
REMOVE_ERRORS    = True      # True: drop error tracebacks from outputs
REMOVE_VIDEOS    = True      # True: drop cells that only display a Video(...)
MAX_IMAGE_WIDTH  = 1600      # shrink big photos (pixels) to keep the PDF small; None = keep originals

TEXT_FONT_PX     = 11        # body text size in pixels (Jupyter default is 14); headings scale with it
CODE_FONT_PX     = 10        # code and output text size (Jupyter default is 13)

## Step 3: Prepare the notebook (embed images, clean up cells)

In [3]:
import base64, copy, io, mimetypes, re
from pathlib import Path
import nbformat

src_path = Path(INPUT_NOTEBOOK).resolve()
assert src_path.exists(), f"Can't find {src_path}. Is this notebook in the same folder?"
base_dir = src_path.parent
out_path = Path(OUTPUT_PDF) if OUTPUT_PDF else src_path.with_suffix(".pdf")

nb = nbformat.read(src_path, as_version=4)
nb = copy.deepcopy(nb)

def to_data_uri(rel_path):
    """Read an image file and return it as a base64 data URI (optionally downscaled)."""
    p = (base_dir / rel_path).resolve()
    if not p.exists():
        print(f"  ! image not found: {rel_path}")
        return None
    mime = mimetypes.guess_type(p.name)[0] or "image/png"
    data = p.read_bytes()
    if MAX_IMAGE_WIDTH and mime in ("image/jpeg", "image/png"):
        try:
            from PIL import Image, ImageOps
            im = ImageOps.exif_transpose(Image.open(io.BytesIO(data)))  # keep phone-photo rotation
            if im.width > MAX_IMAGE_WIDTH:
                im = im.resize((MAX_IMAGE_WIDTH, round(im.height * MAX_IMAGE_WIDTH / im.width)))
            buf = io.BytesIO()
            if mime == "image/jpeg":
                im.convert("RGB").save(buf, "JPEG", quality=85)
            else:
                im.save(buf, "PNG")
            data = buf.getvalue()
        except Exception as e:
            print(f"  (could not resize {rel_path}: {e})")
    return f"data:{mime};base64,{base64.b64encode(data).decode()}"

def is_local(path):
    return not re.match(r"^(https?:|data:|attachment:)", path)

html_img = re.compile(r'(<img\b[^>]*?\bsrc\s*=\s*)(["\'])(.*?)\2', re.IGNORECASE)
md_img   = re.compile(r'(!\[[^\]]*\]\()([^)\s]+)((?:\s+"[^"]*")?\))')

def embed(text):
    def fix_html(m):
        uri = to_data_uri(m.group(3)) if is_local(m.group(3)) else None
        return f"{m.group(1)}{m.group(2)}{uri}{m.group(2)}" if uri else m.group(0)
    def fix_md(m):
        uri = to_data_uri(m.group(2)) if is_local(m.group(2)) else None
        return f"{m.group(1)}{uri}{m.group(3)}" if uri else m.group(0)
    return md_img.sub(fix_md, html_img.sub(fix_html, text))

kept = []
for cell in nb.cells:
    if cell.cell_type == "markdown":
        cell.source = embed(cell.source)
    elif cell.cell_type == "code":
        if REMOVE_VIDEOS and re.fullmatch(r"\s*(from IPython\.display import Video\s*)?Video\(.*\)\s*", cell.source, re.S):
            continue
        if REMOVE_ERRORS:
            cell.outputs = [o for o in cell.get("outputs", []) if o.get("output_type") != "error"]
    kept.append(cell)
# Smaller fonts: a markdown cell holding a <style> block that overrides Jupyter's font-size variables
font_css = f"""<style>
:root {{
  --jp-content-font-size1: {TEXT_FONT_PX}px;
  --jp-ui-font-size1: {TEXT_FONT_PX}px;
  --jp-code-font-size: {CODE_FONT_PX}px;
}}
body {{ font-size: {TEXT_FONT_PX}px; }}
</style>"""
style_cell = nbformat.v4.new_markdown_cell(font_css)
kept.insert(0, style_cell)

nb.cells = kept
print(f"Prepared {len(kept)} cells from {src_path.name}")

Prepared 33 cells from Lab1 Microscopy 2026.ipynb


## Step 4: Export to PDF

In [4]:
from nbconvert import WebPDFExporter, HTMLExporter

try:
    exporter = WebPDFExporter()
    exporter.exclude_input = HIDE_CODE
    exporter.exclude_input_prompt = True
    exporter.exclude_output_prompt = True
    exporter.allow_chromium_download = True
    exporter.page_render_timeout = 5000   # ms: gives MathJax time to typeset equations
    pdf_bytes, _ = exporter.from_notebook_node(nb, resources={"metadata": {"path": str(base_dir)}})
    out_path.write_bytes(pdf_bytes)
    print(f"Saved PDF: {out_path}")
except Exception as e:
    # Fallback: save a self-contained HTML file you can open and print to PDF from your browser
    print(f"PDF export failed ({type(e).__name__}: {e})")
    html_exporter = HTMLExporter()
    html_exporter.exclude_input = HIDE_CODE
    html_exporter.exclude_input_prompt = True
    html_exporter.exclude_output_prompt = True
    body, _ = html_exporter.from_notebook_node(nb)
    html_path = out_path.with_suffix(".html")
    html_path.write_text(body, encoding="utf-8")
    print(f"Saved HTML instead: {html_path}\nOpen it in Chrome and use File → Print → Save as PDF.")

Saved PDF: /Users/hyunseo.lee/Library/CloudStorage/OneDrive-JohnsHopkins/JHU_courses/modern_optics/Lab 1/Lab1 Microscopy 2026.pdf


## Troubleshooting

- **"image not found"**: the path in the `<img src="...">` doesn't match a file. Check the spelling and capitalization (`.JPG` vs `.jpg` matters on some systems) and that this notebook is in the same folder as your report.
- **Chromium download fails**: run `!python -m playwright install chromium` again, or use the HTML fallback that Step 4 saves automatically.
- **Equations missing in the PDF**: the renderer loads MathJax from the internet, so make sure you're online while exporting. If equations still show as raw `$$...$$`, raise `exporter.page_render_timeout` in Step 4 (e.g. to 10000).
- **Text too big or too small?** Change `TEXT_FONT_PX` and `CODE_FONT_PX` in Step 2 and re-run Steps 2–4. Headings and tables scale with the text size. Titles written with `<font size="+3">` keep their own size.
- **Want the code shown?** Set `HIDE_CODE = False` in Step 2 and re-run Steps 2–4.